##設定層數與神經元數

至少需要**2層**深度學習

In [ ]:
N1 = 20
N2 = 30

可選擇是否要啟用第3第4層

In [ ]:
N3 = 20  #可選

In [ ]:
N4 = 20  #可選

可選擇停用第3第4層 (N3,N4設為0)

In [ ]:
N3 = 0  #可選

In [ ]:
N4 = 0  #可選

點擊 **全部執行** 按鈕則為**2層**深度學習



---



## 1. 讀入套件

In [ ]:
!pip install gradio

In [ ]:
%matplotlib inline

# 標準數據分析、畫圖套件
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

# 神經網路方面
import tensorflow as tf
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import SGD

# 互動設計用
from ipywidgets import interact_manual

# 神速打造 web app 的 Gradio
import gradio as gr

## 2. 讀入 MNIST 數據庫與格式整理

訓練資料`60000`筆，測試資料`10000`筆。

In [ ]:
(x_train, y_train), (x_test, y_test) = mnist.load_data()

將輸入格式做**normalization**

In [ ]:
x_train = x_train.reshape(60000, 784)/255
x_test = x_test.reshape(10000, 784)/255

將輸出格式做**1-hot enconding**

In [ ]:
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

## 3. 打造神經網路

我們決定了我們的函數是

$$\hat{f} \colon \mathbb{R}^{784} \to \mathbb{R}^{10}$$

###建構神經網路

In [ ]:
model = Sequential()

前兩層分別有`20`,`30`個神經元，激發函數為`relu`

最後一層有`10`個神經元，激發函數為`softmax`

In [ ]:
model.add(Dense(N1, input_dim=784, activation='relu'))

In [ ]:
model.add(Dense(N2, activation='relu'))

In [ ]:
try:
    if N3 > 0:
        model.add(Dense(N3, activation='relu'))
except:
    pass

In [ ]:
try:
    if N4 > 0:
        model.add(Dense(N4, activation='relu'))
except:
    pass

In [ ]:
model.add(Dense(10, activation='softmax'))

###組裝

我們還要做 `compile`

* 決定使用的 loss function, 一般是 `mse`
* 決定 optimizer, 我們用標準的 SGD
* 設 learning rate 為`0.4`

In [ ]:
model.compile(loss='mse', optimizer=SGD(learning_rate=0.4), metrics=['accuracy'])

## 4. 檢視神經網路

model 的 summary

In [ ]:
model.summary()

## 5. 訓練神經網路

* 一次要訓練`32`筆資料
* 一共要訓練`30`次

In [ ]:
model.fit(x_train, y_train, batch_size=32, epochs=30)

## 6. 測試結果

In [ ]:
loss, acc = model.evaluate(x_test, y_test)

In [ ]:
predict = np.argmax(model.predict(x_test), axis=-1)

In [ ]:
def test(測試編號):
    plt.imshow(x_test[測試編號].reshape(28,28), cmap='Greys')
    print('神經網路判斷為:', predict[測試編號])

In [ ]:
score = model.evaluate(x_test, y_test)

互動操作、計算loss與正確率

In [ ]:
interact_manual(test, 測試編號=(0, 9999));

In [ ]:
print('loss:', score[0])
print('正確率', score[1])

操作變因為：深度學習層數、神經元數、激發函數(以上都不包含激發函數為`softmax`的最後一層)、learning rate、一次訓練資料數(`batch_size`)、訓練次數(`epochs`)

控制變因為：其他所有變因

應變變因：正確率、loss

**先假設二次以上交互項的影響相比一次影響可忽略**

測試深度學習層數與正確率的關係
| 深度學習層數 | 正確率 | loss |
| ----------- | ----------- | ----------- |
| 2 | 0.8960 | 0.0162 |
| 3 | 0.8935 | 0.0161 |
| 4 | 0.8732 | 0.0193 |

2層與3層差距不大，4層正確率降低且loss升高，接下來以正確率最高的2層做神經元數的探討

| N1神經元數 | N2神經元數 | 正確率 | loss |
| ----------- | ----------- | ----------- | ----------- |
| 10 | 10 | 0.8266 | 0.0262 |
| 20 | 10 | 0.8622 | 0.0214 |
| 30 | 10 | 0.8977 | 0.0162 |
| 10 | 20 | 0.8525 | 0.0231 |
| 20 | 20 | 0.8999 | 0.0159 |
| 30 | 20 | 0.9127 | 0.0137 |
| 10 | 30 | 0.8801 | 0.0183 |
| 20 | 30 | 0.9069 | 0.0145 |
| 30 | 30 | 0.9025 | 0.0152 |

實驗數據表示(`N1`,`N2`)=(`20`,`30`)是最佳組合，可能因實驗次數過少(導致隨機性)或假設嚴重錯誤(高次影響實際不小)導致，但差距依然不大

接著使用(`N1`,`N2`)=(`20`,`30`)的組合測試激發函數relu,sigmoid

| N1激發函數 | N2激發函數 | 正確率 | loss |
| ----------- | ----------- | ----------- | ----------- |
| relu | relu | 0.9069 | 0.0145 |
| relu | sigmoid | 0.6447 | 0.0612 |
| sigmoid | relu | 0.5121 | 0.0668 |
| sigmoid | sigmoid | 0.2334 | 0.0888 |

`relu`較`sigmoid`好，且先用`sigmoid`正確率更低。

learning rate與正確率和loss的關係(每組測量三次取平均值)
| learning rate | 正確率 | loss |
| ----------- | ----------- | ----------- |
| 0.1 | 0.9079±0.0015 | 0.0143±0.0002 |
| 0.087 | 0.9048±0.0004 | 0.0149±0.0003 |
| 0.040 | 0.827±0.024 | 0.0275±0.0031 |

針對這個專案，learning rate越大越好
測試`0.4`的正確率更高(約`0.94`)。

使用learning rate 0.4做`batch_size`與`epochs`的測試
| batch_size | epochs | 正確率 | loss |
| ----------- | ----------- | ----------- | ----------- |
| 32 | 10 | 0.9515 | 0.0075 |
| 32 | 30 | 0.9616 | 0.0061 |
| 100 | 10 | 0.9286 | 0.0096 |
| 100 | 30 | 0.9554 | 0.0069 |
| 256 | 10 | 0.9168 | 0.0127 |
| 256 | 30 | 0.9392 | 0.0093 |

`epochs=30`都比`epochs=10`好

`batch_size=32`最佳

得出結果：`2`層深度學習，神經元`N1=20`,`N2=30`，激發函數為`relu`，`learning rate=0.4`，`batch_size=32`，`epochs=30`為現階段測試最佳結果。

In [ ]:
def resize_image(inp):
    # 圖在 inp["layers"][0]
    image = np.array(inp["layers"][0], dtype=np.float32)
    image = image.astype(np.uint8)

    # 轉成 PIL 格式
    image_pil = Image.fromarray(image)

    # Alpha 通道設為白色, 再把圖從 RGBA 轉成 RGB
    background = Image.new("RGB", image_pil.size, (255, 255, 255))
    background.paste(image_pil, mask=image_pil.split()[3]) # 把圖片粘貼到白色背景上，使用透明通道作為遮罩
    image_pil = background

    # 轉換為灰階圖像
    image_gray = image_pil.convert("L")

    # 將灰階圖像縮放到 28x28, 轉回 numpy array
    img_array = np.array(image_gray.resize((28, 28), resample=Image.LANCZOS))

    # 配合 MNIST 數據集
    img_array = 255 - img_array

    # 拉平並縮放
    img_array = img_array.reshape(1, 784) / 255.0

    return img_array

In [ ]:
def recognize_digit(inp):
    img_array = resize_image(inp)
    prediction = model.predict(img_array).flatten()
    labels = list('0123456789')
    return {labels[i]: float(prediction[i]) for i in range(10)}

In [ ]:
iface = gr.Interface(
    fn=recognize_digit,
    inputs=gr.Sketchpad(),
    outputs=gr.Label(num_top_classes=3),
    title="MNIST 手寫辨識",
    description="請在畫板上繪製數字"
)

iface.launch(share=True, debug=True)

> Embedded output screenshot removed from the portfolio notebook; see the submitted PDF for the original result.